# Devoir 1 : ETE435 Géoinformatique
## Ingénierie des données géospatiales : classe `Aeroports`, LineString, Polygon et analyse de proximité

**Préparé par :**  
- Alan CASSEUS, Mémorant en Sciences de l'Environnement (FSTEAT/CHCL-UEH)  
- Joscelyn PARFAIT, Diplômé en Génie Civil (FSG/CHCL-UEH)

**Soumis au Professeur :** Paul Célicourt, PhD en Hydroinformatique  
**Date de soumission :** 29 Juin 2026


## Partie 0 : Configuration de l'environnement

Avant d'écrire le moindre code, on a créé un environnement Python virtuel dédié a isoler les dépendances du projet et éviter tout conflit avec d'autres installations. Cela permet aussi de reproduire exactement les mêmes résultats sur une autre machine.

```bash
# Création de l'environnement virtuel
python -m venv venv_devoir1

# Activation (Windows)
venv_devoir1\Scripts\activate

# Installation des bibliothèques
pip install pandas geopandas shapely folium matplotlib numpy pyproj
pip install notebook jupyterlab

# Export de la configuration
pip freeze > requirements.txt
```

In [1]:
# Traçabilité de l'environnement
import sys, platform
import pandas as pd
import geopandas as gpd
import shapely
import numpy as np
import folium

print(f"Python        : {sys.version.split()[0]}")
print(f"pandas        : {pd.__version__}")
print(f"geopandas     : {gpd.__version__}")
print(f"shapely       : {shapely.__version__}")
print(f"numpy         : {np.__version__}")
print(f"folium        : {folium.__version__}")
print(f"Plateforme    : {platform.system()} {platform.release()}")

Python        : 3.13.14
pandas        : 3.0.1
geopandas     : 1.1.3
shapely       : 2.1.2
numpy         : 2.4.3
folium        : 0.20.0
Plateforme    : Windows 11


## Partie 1 : Audit et nettoyage des données

Avant tout traitement géospatial, on fait un audit du fichier brut `airports_canada_chcl.csv` pour repérer et corriger les anomalies. C'est une étape qu'on ne peut pas sauter : des coordonnées manquantes ou incohérentes donneraient des résultats faux sans lever la moindre erreur.

On distingue trois types de problèmes dans ce fichier :
1. Les valeurs manquantes (NaN natifs) dans les colonnes latitude ou longitude
2. Les valeurs non numériques, c'est-à-dire du texte parasite dans la colonne longitude
3. Les valeurs aberrantes, soit des coordonnées géographiquement impossibles ou situées hors du Canada

In [2]:
import warnings
import math
import random
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString, Polygon
import folium

warnings.filterwarnings('ignore')

# 1.1 Chargement et inspection initiale 
CSV_PATH = 'airports_canada_chcl.csv'
df_raw = pd.read_csv(CSV_PATH)

print("=" * 55)
print("INSPECTION INITIALE")
print("=" * 55)
print(f"Nombre de lignes initial : {len(df_raw)}")
print(f"\nTypes des colonnes clés :")
print(df_raw[['latitude_deg', 'longitude_deg']].dtypes)
print(f"\nValeurs NaN natives :")
print(df_raw[['latitude_deg', 'longitude_deg']].isna().sum())

INSPECTION INITIALE
Nombre de lignes initial : 417

Types des colonnes clés :
latitude_deg     float64
longitude_deg        str
dtype: object

Valeurs NaN natives :
latitude_deg     37
longitude_deg    37
dtype: int64


In [3]:
# 1.2 Détection des valeurs non numériques 
# La colonne longitude_deg est de type 'str' (object), ce qui indique
# la présence de valeurs textuelles parasites mêlées aux coordonnées.

lon_non_num = df_raw[
    pd.to_numeric(df_raw['longitude_deg'], errors='coerce').isna()
    & df_raw['longitude_deg'].notna()
]
print("Longitudes non numériques détectées :")
for _, r in lon_non_num.iterrows():
    print(f"  ident={r['ident']:<6s}  valeur='{r['longitude_deg']}'  nom={r['name']}")
print(f"\n→ {len(lon_non_num)} ligne(s) avec longitude non numérique")

Longitudes non numériques détectées :
  ident=CEL5    valeur='CEL15'  nom=Valleyview Airport
  ident=CAZ5    valeur='ETE435'  nom=Cache Creek-Ashcroft Regional Airport

→ 2 ligne(s) avec longitude non numérique


In [4]:
# 1.3 Conversion robuste des types 
# errors='coerce' : les valeurs non convertibles deviennent NaN silencieusement,
# évitant ainsi un arrêt brutal du script.
df = df_raw.copy()
df['latitude_deg']  = pd.to_numeric(df['latitude_deg'],  errors='coerce')
df['longitude_deg'] = pd.to_numeric(df['longitude_deg'], errors='coerce')

# 1.4 Suppression des lignes avec coordonnées manquantes 
df_clean = df.dropna(subset=['latitude_deg', 'longitude_deg']).copy()
n_removed_na = len(df_raw) - len(df_clean)
print(f"Lignes supprimées (NaN / non-numérique) : {n_removed_na}")
print(f"Lignes restantes après dropna            : {len(df_clean)}")

Lignes supprimées (NaN / non-numérique) : 58
Lignes restantes après dropna            : 359


In [5]:
# 1.5 Détection et suppression des valeurs aberrantes 
# Bornes géographiques retenues pour le Canada :
#   Latitude  : [40°N, 84°N]  (du lac Érié à l'archipel arctique)
#   Longitude : [-141°E, -50°E] (du Yukon/Alaska à Terre-Neuve)

mask_aberrant = (
    (df_clean['latitude_deg']  < 40)   |
    (df_clean['latitude_deg']  > 84)   |
    (df_clean['longitude_deg'] < -141) |
    (df_clean['longitude_deg'] > -50)
)
df_aberrant = df_clean[mask_aberrant].copy()

print(f"Lignes aberrantes détectées : {len(df_aberrant)}")
print(df_aberrant[['ident', 'name', 'latitude_deg', 'longitude_deg']].to_string(index=False))

df_clean = df_clean[~mask_aberrant].reset_index(drop=True)
print(f"\nLignes valides finales : {len(df_clean)}")

Lignes aberrantes détectées : 7
ident                                           name  latitude_deg  longitude_deg
 CYVR                Vancouver International Airport    -50.567000    -123.183998
 CYOW Ottawa Macdonald-Cartier International Airport    -50.567000     -75.669197
 CYHZ      Halifax / Stanfield International Airport    -50.567000     -63.508598
 CYYT               St. John's International Airport    -50.567000     -52.751900
 CYQA                                Muskoka Airport    -50.567000     -79.306546
 CYFH                              Fort Hope Airport     51.561901    -150.000000
 CYGZ                            Grise Fiord Airport     92.567680     -82.908583

Lignes valides finales : 352


In [6]:
# 1.6 Résumé du nettoyage 
n_initial   = len(df_raw)
n_removed   = n_removed_na
n_aberrant  = len(df_aberrant)
n_valid     = len(df_clean)

resume = pd.DataFrame([
    {'Étape': 'Lignes initiales (fichier brut)', 'Lignes': n_initial, 'Action': '—'},
    {'Étape': 'Coordonnées manquantes (NaN natifs) + non-numériques', 'Lignes': n_removed, 'Action': 'Supprimées'},
    {'Étape': 'Valeurs aberrantes géographiques', 'Lignes': n_aberrant, 'Action': 'Supprimées'},
    {'Étape': 'Lignes valides restantes (airports_clean.csv)', 'Lignes': n_valid, 'Action': 'Conservées'},
])
print(resume.to_string(index=False))

# Export du jeu de données nettoyé
df_clean.to_csv('airports_clean.csv', index=False)
print(f"\n✓ airports_clean.csv exporté ({n_valid} lignes)")

                                               Étape  Lignes     Action
                     Lignes initiales (fichier brut)     417          —
Coordonnées manquantes (NaN natifs) + non-numériques      58 Supprimées
                    Valeurs aberrantes géographiques       7 Supprimées
       Lignes valides restantes (airports_clean.csv)     352 Conservées


✓ airports_clean.csv exporté (352 lignes)


## Partie 2 : Classe `Aeroports` : définition et instanciation

La classe `Aeroports` regroupe tout le pipeline de traitement géospatial. Elle est organisée autour de quatre méthodes principales qui correspondent aux quatre étapes du devoir :

- `preparer_donnees()` : chargement, nettoyage et validation des coordonnées
- `creer_linestring()` : sélection de 3 aéroports et construction du LineString
- `creer_polygon()` : construction du Polygon à partir des mêmes aéroports
- `analyser_proximite()` : sélection aléatoire d'un aéroport et test d'appartenance au polygone

In [7]:
class Aeroports:
    """Classe de traitement et d'analyse géospatiale des aéroports canadiens."""

    # Bornes géographiques du Canada
    LAT_MIN, LAT_MAX = 40.0, 84.0
    LON_MIN, LON_MAX = -141.0, -50.0

    def __init__(self, filepath: str):
        self.filepath = filepath
        self.df_raw   = None
        self.df_clean = None
        self.selected_airports = None
        self.linestring = None
        self.polygon    = None

    # Étape 1 : Préparation des données
    def preparer_donnees(self) -> pd.DataFrame:
        """Charge, nettoie et valide les coordonnées géographiques."""
        self.df_raw = pd.read_csv(self.filepath)
        df = self.df_raw.copy()

        # Conversion robuste des types
        df['latitude_deg']  = pd.to_numeric(df['latitude_deg'],  errors='coerce')
        df['longitude_deg'] = pd.to_numeric(df['longitude_deg'], errors='coerce')

        # Suppression des coordonnées manquantes ou invalides
        df = df.dropna(subset=['latitude_deg', 'longitude_deg'])

        # Suppression des valeurs aberrantes hors des limites du Canada
        mask = (
            (df['latitude_deg'].between(self.LAT_MIN, self.LAT_MAX)) &
            (df['longitude_deg'].between(self.LON_MIN, self.LON_MAX))
        )
        self.df_clean = df[mask].reset_index(drop=True)

        n_initial = len(self.df_raw)
        n_valid   = len(self.df_clean)
        print(f"Données chargées        : {n_initial} lignes")
        print(f"Lignes supprimées       : {n_initial - n_valid}")
        print(f"Données valides         : {n_valid} lignes")
        return self.df_clean

    # Étape 2 : Création du LineString 
    def creer_linestring(self, idents: list = None) -> LineString:
        """Sélectionne automatiquement 3 aéroports et crée un LineString."""
        if self.df_clean is None:
            self.preparer_donnees()

        # Sélection automatique : grands aéroports couvrant le corridor est-ouest
        if idents is None:
            idents = ['CYUL', 'CYYZ', 'CYYC']

        self.selected_airports = (
            self.df_clean[self.df_clean['ident'].isin(idents)]
            .set_index('ident').loc[idents].reset_index()
        )

        coords = [
            (r['longitude_deg'], r['latitude_deg'])
            for _, r in self.selected_airports.iterrows()
        ]
        self.linestring = LineString(coords)

        print("Aéroports sélectionnés :")
        for _, r in self.selected_airports.iterrows():
            print(f"  {r['ident']} — {r['name'][:50]}")
            print(f"          lat={r['latitude_deg']:.4f}°, lon={r['longitude_deg']:.4f}°")
        print(f"\nLineString créé (longueur euclidienne : {self.linestring.length:.4f}°)")
        return self.linestring

    def afficher_linestring(self) -> folium.Map:
        """Affiche le LineString sur une carte Folium."""
        centre = [
            self.selected_airports['latitude_deg'].mean(),
            self.selected_airports['longitude_deg'].mean()
        ]
        m = folium.Map(location=centre, zoom_start=4, tiles='CartoDB positron')

        # Tracé du LineString
        locs = [(r['latitude_deg'], r['longitude_deg'])
                for _, r in self.selected_airports.iterrows()]
        folium.PolyLine(locs, color='steelblue', weight=3,
                        tooltip='Parcours LineString').add_to(m)

        # Marqueurs pour chaque aéroport
        for _, r in self.selected_airports.iterrows():
            folium.Marker(
                location=[r['latitude_deg'], r['longitude_deg']],
                popup=f"{r['ident']} — {r['name']}",
                icon=folium.Icon(color='blue', icon='plane', prefix='fa')
            ).add_to(m)

        return m

    # Étape 3 : Création du Polygon
    def creer_polygon(self) -> Polygon:
        """Construit un Polygon à partir des mêmes aéroports que le LineString."""
        if self.selected_airports is None:
            self.creer_linestring()

        coords = [
            (r['longitude_deg'], r['latitude_deg'])
            for _, r in self.selected_airports.iterrows()
        ]
        self.polygon = Polygon(coords)

        print(f"Polygon créé :")
        print(f"  Surface     : {self.polygon.area:.4f} degrés²")
        print(f"  Périmètre   : {self.polygon.length:.4f} degrés")
        print(f"  Valide      : {self.polygon.is_valid}")
        return self.polygon

    def afficher_polygon(self) -> folium.Map:
        """Affiche le Polygon sur une carte Folium."""
        centre = [
            self.selected_airports['latitude_deg'].mean(),
            self.selected_airports['longitude_deg'].mean()
        ]
        m = folium.Map(location=centre, zoom_start=4, tiles='CartoDB positron')

        # Tracé du Polygon
        poly_locs = [(r['latitude_deg'], r['longitude_deg'])
                     for _, r in self.selected_airports.iterrows()]
        folium.Polygon(
            locations=poly_locs,
            color='steelblue', fill=True, fill_color='lightblue',
            fill_opacity=0.4, weight=2,
            tooltip='Zone polygonale'
        ).add_to(m)

        # Marqueurs sommets
        for _, r in self.selected_airports.iterrows():
            folium.Marker(
                location=[r['latitude_deg'], r['longitude_deg']],
                popup=f"{r['ident']} — {r['name']}",
                icon=folium.Icon(color='blue', icon='plane', prefix='fa')
            ).add_to(m)

        return m

    # Étape 4 : Analyse spatiale 
    def analyser_proximite(self, seed: int = 42) -> dict:
        """Sélectionne un aéroport aléatoire et vérifie s'il est dans le polygone."""
        if self.polygon is None:
            self.creer_polygon()

        # Exclure les 3 aéroports déjà utilisés
        idents_exclus = list(self.selected_airports['ident'])
        candidats = self.df_clean[~self.df_clean['ident'].isin(idents_exclus)].copy()

        random.seed(seed)
        idx = random.randint(0, len(candidats) - 1)
        aeroport = candidats.iloc[idx]

        pt     = Point(aeroport['longitude_deg'], aeroport['latitude_deg'])
        inside = self.polygon.contains(pt)

        result = {
            'ident'    : aeroport['ident'],
            'nom'      : aeroport['name'],
            'latitude' : aeroport['latitude_deg'],
            'longitude': aeroport['longitude_deg'],
            'interieur': inside,
        }

        print("\n=== Résultat de l'analyse de proximité ===")
        print(f"  Aéroport aléatoire : {result['ident']} — {result['nom']}")
        print(f"  Position           : lat={result['latitude']:.4f}°, lon={result['longitude']:.4f}°")
        statut = "INTÉRIEUR" if inside else "EXTÉRIEUR"
        print(f"  Résultat           : {statut} du polygone")
        return result

    def afficher_analyse(self, result: dict) -> folium.Map:
        """Affiche le polygone et l'aéroport aléatoire sur une carte."""
        centre = [
            self.selected_airports['latitude_deg'].mean(),
            self.selected_airports['longitude_deg'].mean()
        ]
        m = folium.Map(location=centre, zoom_start=4, tiles='CartoDB positron')

        # Polygone
        poly_locs = [(r['latitude_deg'], r['longitude_deg'])
                     for _, r in self.selected_airports.iterrows()]
        folium.Polygon(
            locations=poly_locs,
            color='steelblue', fill=True, fill_color='lightblue',
            fill_opacity=0.3, weight=2, tooltip='Zone polygonale'
        ).add_to(m)

        # Aéroports sommets
        for _, r in self.selected_airports.iterrows():
            folium.Marker(
                location=[r['latitude_deg'], r['longitude_deg']],
                popup=f"{r['ident']} — {r['name']}",
                icon=folium.Icon(color='blue', icon='plane', prefix='fa')
            ).add_to(m)

        # Aéroport aléatoire
        couleur = 'green' if result['interieur'] else 'red'
        statut  = 'INTÉRIEUR' if result['interieur'] else 'EXTÉRIEUR'
        folium.Marker(
            location=[result['latitude'], result['longitude']],
            popup=f"{result['ident']} — {result['nom']}<br>{statut} du polygone",
            icon=folium.Icon(color=couleur, icon='question-sign')
        ).add_to(m)

        return m

print("Classe Aeroports définie avec succès.")

Classe Aeroports définie avec succès.


## Exécution du flux de travail

Les quatre étapes s'enchaînent dans l'ordre, chacune utilisant les résultats de la précédente.

In [8]:
# Instanciation et étape 1 : préparation des données 
aeroports = Aeroports('airports_canada_chcl.csv')
df_clean  = aeroports.preparer_donnees()

Données chargées        : 417 lignes
Lignes supprimées       : 65
Données valides         : 352 lignes


In [9]:
# Étape 2 : création et affichage du LineString
# On choisit CYUL (Montréal), CYYZ (Toronto) et CYYC (Calgary) parce qu'ils
# forment un corridor est-ouest assez représentatif du territoire canadien.
linestring = aeroports.creer_linestring(idents=['CYUL', 'CYYZ', 'CYYC'])
carte_linestring = aeroports.afficher_linestring()
carte_linestring

Aéroports sélectionnés :
  CYUL — Montreal / Pierre Elliott Trudeau International Ai
          lat=45.4678°, lon=-73.7423°
  CYYZ — Toronto Pearson International Airport
          lat=43.6759°, lon=-79.6294°
  CYYC — Calgary International Airport
          lat=51.1188°, lon=-114.0099°

LineString créé (longueur euclidienne : 41.3307°)


In [10]:
# Étape 3 : création et affichage du Polygon
polygon = aeroports.creer_polygon()
carte_polygon = aeroports.afficher_polygon()
carte_polygon

Polygon créé :
  Surface     : 52.7119 degrés²
  Périmètre   : 81.9929 degrés
  Valide      : True


In [11]:
# Étape 4 : analyse spatiale — point dans polygone
result = aeroports.analyser_proximite(seed=42)
carte_analyse = aeroports.afficher_analyse(result)
carte_analyse


=== Résultat de l'analyse de proximité ===
  Aéroport aléatoire : CYFR — Fort Resolution Airport
  Position           : lat=61.1808°, lon=-113.6900°
  Résultat           : EXTÉRIEUR du polygone


## Export géospatial (GeoJSON)

Les grands aéroports canadiens (`large_airport`) du jeu de données nettoyé sont exportés en GeoJSON avec le système de référence EPSG:4326 (WGS84), ce qui assure la compatibilité avec la plupart des logiciels SIG.

In [12]:
# Export airports_clean.csv
df_clean.to_csv('airports_clean.csv', index=False)
print(f"airports_clean.csv exporté ({len(df_clean)} lignes)")

# Export large_airports_canada.geojson
df_large = df_clean[df_clean['type'] == 'large_airport'].copy()
df_large['geometry'] = df_large.apply(
    lambda r: Point(r['longitude_deg'], r['latitude_deg']), axis=1
)
gdf = gpd.GeoDataFrame(df_large, geometry='geometry').set_crs(epsg=4326)
gdf.to_file('large_airports_canada.geojson', driver='GeoJSON')
print(f"large_airports_canada.geojson exporté ({len(df_large)} grands aéroports)")

# Résumé final
print("\n" + "=" * 45)
print("RÉSUMÉ FINAL DU TRAITEMENT")
print("=" * 45)
print(f"Lignes initiales             : {len(aeroports.df_raw)}")
print(f"Lignes supprimées            : {len(aeroports.df_raw) - len(df_clean)}")
print(f"Lignes valides               : {len(df_clean)}")
print(f"Grands aéroports exportés    : {len(df_large)}")
print(f"LineString (3 aéroports)     : {aeroports.linestring.length:.4f}°")
print(f"Polygon (surface)            : {aeroports.polygon.area:.4f}°²")
print(f"Aéroport aléatoire           : {result['ident']} — {result['nom'][:40]}")
print(f"Résultat analyse             : {'INTÉRIEUR' if result['interieur'] else 'EXTÉRIEUR'} du polygone")

airports_clean.csv exporté (352 lignes)


large_airports_canada.geojson exporté (20 grands aéroports)

RÉSUMÉ FINAL DU TRAITEMENT
Lignes initiales             : 417
Lignes supprimées            : 65
Lignes valides               : 352
Grands aéroports exportés    : 20
LineString (3 aéroports)     : 41.3307°
Polygon (surface)            : 52.7119°²
Aéroport aléatoire           : CYFR — Fort Resolution Airport
Résultat analyse             : EXTÉRIEUR du polygone
